In [3]:

# Install required libraries (if needed)
!pip install pandas numpy scikit-learn

# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from math import sqrt

# ================================
# 2. LOAD DATASET
# ================================
# Using MovieLens small dataset
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"

columns = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(url, sep='\t', names=columns)

print("Dataset Shape:", df.shape)
df.head()

# ================================
# 3. CREATE USER-ITEM MATRIX
# ================================
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating')

# Fill missing values with 0
user_item_matrix_filled = user_item_matrix.fillna(0)

print("Matrix Shape:", user_item_matrix_filled.shape)

# ================================
# 4. COMPUTE SIMILARITY (ITEM-BASED)
# ================================
item_similarity = cosine_similarity(user_item_matrix_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix_filled.columns,
    columns=user_item_matrix_filled.columns
)

print("Item Similarity Matrix Created")

# ================================
# 5. RECOMMENDATION FUNCTION
# ================================
def recommend_items(user_id, num_recommendations=5):
    user_ratings = user_item_matrix_filled.loc[user_id]

    # Get items the user has already rated
    rated_items = user_ratings[user_ratings > 0].index.tolist()

    scores = {}

    for item in rated_items:
        similar_items = item_similarity_df[item]

        for sim_item, score in similar_items.items():
            if sim_item not in rated_items:
                if sim_item not in scores:
                    scores[sim_item] = 0
                scores[sim_item] += score * user_ratings[item]

    # Sort recommendations
    recommended_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return recommended_items[:num_recommendations]

# ================================
# 6. TEST RECOMMENDATION
# ================================
user_id = 5
recommendations = recommend_items(user_id)

print(f"\nTop Recommendations for User {user_id}:")
for item, score in recommendations:
    print(f"Item {item} -> Score: {round(score, 2)}")

# ================================
# 7. EVALUATION (RMSE)
# ================================
# Predict ratings using similarity
predictions = user_item_matrix_filled.dot(item_similarity) / np.array([np.abs(item_similarity).sum(axis=1)])

# Flatten matrices for comparison
actual = user_item_matrix_filled.values.flatten()
predicted = predictions.values.flatten()

# Remove zero entries (unrated items)
mask = actual > 0
rmse = sqrt(mean_squared_error(actual[mask], predicted[mask]))

print("\nRMSE:", round(rmse, 4))


Dataset Shape: (100000, 4)
Matrix Shape: (943, 1682)
Item Similarity Matrix Created

Top Recommendations for User 5:
Item 195 -> Score: 217.49
Item 82 -> Score: 215.88
Item 202 -> Score: 211.97
Item 96 -> Score: 211.29
Item 56 -> Score: 207.42

RMSE: 2.9502
